# Per-Sample McNemar's Test

In [1]:
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np

# For EACH of the 5 folds, compute McNemar, then combine p-values (Fisher's method)
from scipy.stats import chi2

trial = 1

comb_loss_pred_path = f"artifacts/moe_minimal_full_aug/trial_{trial}/predictions.npy"
norm_loss_pred_path = "artifacts/artifacts_max_normalLoss_full/trial_14/predictions.npy"
gt_path = f"artifacts/moe_minimal_full_aug/trial_{trial}/ground_truths.npy"


comb_loss_preds = np.load(comb_loss_pred_path, allow_pickle=True)
norm_loss_preds = np.load(norm_loss_pred_path, allow_pickle=True)
y_true = np.load(gt_path, allow_pickle=True)[0,:].astype(int)


p_values = []

for fold in range(5):
    preds_A = comb_loss_preds[fold]  # shape: (198,)
    preds_B = norm_loss_preds[fold]  # shape: (198,)

    correct_A = (preds_A == y_true)
    correct_B = (preds_B == y_true)

    # Build 2x2 contingency table
    # b: A correct, B wrong | c: A wrong, B correct
    b = np.sum(correct_A & ~correct_B)
    c = np.sum(~correct_A & correct_B)

    table = [[np.sum(correct_A & correct_B), b],
             [c, np.sum(~correct_A & ~correct_B)]]

    result = mcnemar(table, exact=False, correction=True)
    p_values.append(result.pvalue)

# Combine p-values across folds using Fisher's method
chi2_stat = -2 * np.sum(np.log(p_values))
combined_p = 1 - chi2.cdf(chi2_stat, df=2 * len(p_values))

print(f"Combined p-value (Fisher's method): {combined_p:.4f}")


Combined p-value (Fisher's method): 0.0490


In [2]:
import numpy as np
from scipy import stats
from src.utils import calculate_accuracies

# Example: accuracy (or any metric) per fold for model A vs model B
scores_A = np.array(calculate_accuracies(comb_loss_preds, y_true))
scores_B = np.array(calculate_accuracies(norm_loss_preds, y_true))

# Paired t-test: tests if the mean difference is significantly != 0
differences = scores_A - scores_B
t_stat, p_value = stats.ttest_rel(scores_A, scores_B)

print(f"Mean difference: {differences.mean():.4f}")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

Mean difference: 0.0212
t-statistic: 1.0830
p-value: 0.3397
